In [42]:
from snowflake.snowpark import Session
from credentials import params

from pathlib import Path
import subprocess
import time
from snowflake.ml.registry import Registry

import os
from dotenv import load_dotenv
import subprocess

In [20]:
session = Session.builder.configs(params).create()

In [4]:
# DA USARE NELLO SCRIPT (file .py)
# PROJECT_ROOT = Path(__file__).resolve().parent    

# DA USARE NEL NOTEBOOK:
PROJECT_ROOT = Path.cwd() 
PROJECT_ROOT

PosixPath('/Users/filippopedrini/PROGRAMMING/snowflake_housingprice_ml_pipeline')

# 0_setup

In [5]:
SetUp = Path('sql/0_setup.sql')

In [6]:
subprocess.run(
    ["snow", "sql", "-f", f"{SetUp}"],
    check=True
)

CREATE WAREHOUSE IF NOT EXISTS HOUSEPRICEPROJECT_WH1;
+--------+
| status |
|--------|
+------------------------------------------------------------+
| status                                                     |
|------------------------------------------------------------|
| HOUSEPRICEPROJECT_WH1 already exists, statement succeeded. |
+------------------------------------------------------------+

CREATE DATABASE IF NOT EXISTS HOUSING_PRICE_PROJECT;
+--------+
| status |
|--------|
+------------------------------------------------------------+
| status                                                     |
|------------------------------------------------------------|
| HOUSING_PRICE_PROJECT already exists, statement succeeded. |
+------------------------------------------------------------+

CREATE SCHEMA IF NOT EXISTS HOUSING_PRICE_PROJECT.STAGING_LAYER;
+--------+
| status |
|--------|
+--------------------------------------------+
| status                                     |
|--

CompletedProcess(args=['snow', 'sql', '-f', 'sql/0_setup.sql'], returncode=0)

# Upload Python Files in Snowflake Stage

In [7]:
path = Path("src/ML")

files = list(path.glob("*.py"))

for file in files:
    session.sql(f"""PUT file://{PROJECT_ROOT / file} 
        @HOUSING_PRICE_PROJECT.PUBLIC.PROJECT_CODE/ML_CODE
        AUTO_COMPRESS=FALSE
        OVERWRITE=TRUE;""").collect()
    print("IMPORTED: ", file)

IMPORTED:  src/ML/ML_hyperp_tune_root.py
IMPORTED:  src/ML/ML_hyperp_tune_func2.py
IMPORTED:  src/ML/ML_traintest_split.py
IMPORTED:  src/ML/ML_hyperp_tune_entrypoint.py
IMPORTED:  src/ML/ML_hyperp_tune.py
IMPORTED:  src/ML/ML_training.py
IMPORTED:  src/ML/ML_preproc.py
IMPORTED:  src/ML/ML_hyperp_tune_test.py
IMPORTED:  src/ML/ML_hyperp_tune_func1.py


In [8]:
path = Path("src/data_cleaning")

files = list(path.glob("*.py"))

for file in files:
    session.sql(f"""PUT file://{PROJECT_ROOT / file} 
        @HOUSING_PRICE_PROJECT.PUBLIC.PROJECT_CODE
        AUTO_COMPRESS=FALSE
        OVERWRITE=TRUE;""").collect()
    print("IMPORTED: ", file)

IMPORTED:  src/data_cleaning/clean_data.py
IMPORTED:  src/data_cleaning/data_quality_report.py


# Create all the remaining infrastructure

In [75]:
path = Path("sql")
files = list(path.glob("[0-9]*.sql"))
files.sort()

files.remove(Path('sql/0_setup.sql')) # remove 0_setup.sql because it was already executed

print("files that will be executed:\n")
for file in files:
    print(file)
print("")

files that will be executed:

sql/1_data_ingestion.sql
sql/2_data_quality_check.sql
sql/3_clean_data.sql
sql/4_ML_traintestsplit.sql
sql/5_ML_preprocessing.sql
sql/6_ML_hyperparam_tuning.sql
sql/7_ML_training.sql



In [10]:
for file in files:
    subprocess.run(
        ["snow", "sql", "-f", f"{file}"],
        check=True
    )

CREATE OR REPLACE FILE FORMAT HOUSING_PRICE_PROJECT.PUBLIC.CSV_INFERSCHEMA_FF
TYPE = CSV
PARSE_HEADER = TRUE
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
NULL_IF = ('NULL','N/A','NA');
+--------+
| status |
|--------|
+------------------------------------------------------+
| status                                               |
|------------------------------------------------------|
| File format CSV_INFERSCHEMA_FF successfully created. |
+------------------------------------------------------+

CREATE TABLE IF NOT EXISTS HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA(
    PROPERTY_ID                  VARCHAR,
    CITY                         VARCHAR,
    LOCALITY                     VARCHAR,
    LOCALITY_TIER                VARCHAR,
    PROPERTY_TYPE                VARCHAR,
    BHK                          NUMBER,
    BATHROOMS                    NUMBER,
    BALCONIES                    NUMBER,
    BUILT_UP_AREA                NUMBER,
    CARPET_AREA     

# Storage Integration

In [ ]:
# load the .env file
load_dotenv()

# read the file storage_integration.sql
sql = Path("sql/storage_integration.sql").read_text()

# replace placeholders with values from .env file
sql = sql.replace("{{AWS_ROLE_ARN}}", os.environ["AWS_ROLE_ARN"])
sql = sql.replace("{{S3_BUCKET_URL}}", os.environ["S3_BUCKET_URL"])

# write the query in a temporary sql file
Path("tmp.sql").write_text(sql)

# execute the sql commands
subprocess.run(
    ["snow", "sql", "-f", "tmp.sql"],
    check=True
)

# remove the file tmp.sql
os.remove("tmp.sql")

In [13]:
desc_integr = session.sql("DESCRIBE INTEGRATION HOUSEPRICE_S3_INTEGRATION").to_pandas()

STORAGE_AWS_IAM_USER_ARN = desc_integr.loc[
    desc_integr['"property"'] == 'STORAGE_AWS_IAM_USER_ARN', '"property_value"'
    ].reset_index(drop=True)[0]

STORAGE_AWS_EXTERNAL_ID = desc_integr.loc[
    desc_integr['"property"'] == 'STORAGE_AWS_EXTERNAL_ID', '"property_value"'
    ].reset_index(drop=True)[0]


def TrustRelCodes(STORAGE_AWS_IAM_USER_ARN, STORAGE_AWS_EXTERNAL_ID):
    print(
        "\nPlease copy the following values and add them to the "
        "Trust Relationship policy of the AWS IAM role used by the S3 bucket:"
    )
    print(f"\nSTORAGE_AWS_IAM_USER_ARN = {STORAGE_AWS_IAM_USER_ARN}")
    print(f"STORAGE_AWS_EXTERNAL_ID = {STORAGE_AWS_EXTERNAL_ID}")
    
    copied = False
    while not copied:
        quest_input = input("\nEnter 'y' once you have copied both values: ")
        if quest_input.lower() == 'y':
            copied = True


ConnectionWorks = False
Attempt = 0

while not ConnectionWorks:
    try: # check if the connection to the S3 bucket works
        
        ListStage = session.sql("LIST @HOUSING_PRICE_PROJECT.PUBLIC.HOUSEPRICE_STAGE;").to_pandas() 
        ConnectionWorks = True
        print(f"Attempt {Attempt}") if Attempt > 0 else 0
        print("connection works!")
        print("\nfiles in S3 bucket:")
        for row in range(ListStage['"name"'].count()):
            print(ListStage.iloc[row,0])
        print("\n" + 80 * "*")
    
    except: # in case the connection doesn't work and "try" failed

        # print it at the first attempt (i.e. when Attempt == 1)
        print(f"Attempt {Attempt}") if Attempt > 0 else 0           
        print("connection does NOT work") if Attempt > 0 else 0

        # runs function that prints the codes for S3-bucket trust relationship, only before any attempt (i.e. when Attempt == 0)
        TrustRelCodes(STORAGE_AWS_IAM_USER_ARN, STORAGE_AWS_EXTERNAL_ID) if Attempt == 0 else 0 

        # after 5 attempts it breaks out of the loop and interrupt the code
        if Attempt == 5:
            ConnectionWorks = True
            print("failed! no attempts left!\n" + 80 * "*")
            break
        
        
        
        Ready = False

        # print this only before first test attempt (when Attempt == 0)
        if Attempt == 0:  
            print("\n" + 80 * "*" + "\nCONNECTION TEST\nPlease, enter 'y' to test the connection, 'n' to interrupt it\n" + 80 * "*")
            re = ""
        else:
            re = "re-" 
   
        #  query user whether to start or not the test, which means let while-loop to run into the "try" statement above
        while not Ready:
            
            quest_input = input(f"\nProceed with connection {re}test: ") # query the user whether to start the test or not
            
            if quest_input.lower() == 'y': # verify whether the user says "yes"
                Attempt += 1
                Ready = True 
            
            elif quest_input.lower() == 'n': # verify whether the user says "no"
                print("interrupt testing\n\n" + 80 * "*")
                Ready = 7 # assign this to Ready, to trigger the coming if-clause 

        if Ready == 7: # interrupting main while-loop due to negative user answer on running connection test question
            break

connection works!

files in S3 bucket:
s3://snowflake-housingprice-ml-pipeline-project/HousingPricePredictionDataset_train.csv

********************************************************************************


# First Task Execution

In [14]:
sql_ingest = """
COPY INTO HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA
FROM @HOUSING_PRICE_PROJECT.PUBLIC.HOUSEPRICE_STAGE
FILE_FORMAT = 'HOUSING_PRICE_PROJECT.PUBLIC.CSV_INFERSCHEMA_FF'
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
INCLUDE_METADATA = (
    SOURCE_FILE = METADATA$FILENAME
);
"""
print(sql_ingest.strip())

COPY INTO HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA
FROM @HOUSING_PRICE_PROJECT.PUBLIC.HOUSEPRICE_STAGE
FILE_FORMAT = 'HOUSING_PRICE_PROJECT.PUBLIC.CSV_INFERSCHEMA_FF'
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
INCLUDE_METADATA = (
    SOURCE_FILE = METADATA$FILENAME
);


In [15]:
actions_and_checks = [
    ("sql_ingest", "STAGING_LAYER.RAW_DATA"),
    ("STAGING_LAYER.QUALITY_CHECK_REPORT_PROC()", "STAGING_LAYER.DATA_QUALITY_REPORT"),
    ("CLEANING_LAYER.CLEAN_DATA_PROC(10)", "CLEANING_LAYER.CLEAN_DATA"),
    ("ML_LAYER.TRAINTEST_SPLIT_PROC(5000)", "ML_LAYER.TRAIN_SET"),
    ("ML_LAYER.PREPROCESSING_PROC()", "model_registry_check"),
    ("ML_LAYER.HYPERPARAM_TUNING_PROC()", "ML_LAYER.XGBOOST_BEST_PARAMETERS"),
    ("ML_LAYER.XGBOOST_TRAINING_PROC()", "ML_LAYER.XGBOOST_TRAINING_RESULTS")
]

In [72]:
def TaskSuccessful(check):
    cicle_time_s = 10

    output = 0
    nr_cicle = 0 
    while output == 0:
        
        if nr_cicle == 3:
            print(f"{check} tab is either empty or not accessible!")
            return False
        
        try:    
            query = f"SELECT COUNT(1) FROM HOUSING_PRICE_PROJECT.{check};"
            output = PreprocModelRegistryCheck() if check == "model_registry_check" else session.sql(query).collect()[0][0]
            print(output)
        except:
            output = 0
        
        time.sleep(cicle_time_s)
        
        nr_cicle += 1
    
    return True




def PreprocModelRegistryCheck():

    registry = Registry(
        session=session,
        database_name="HOUSING_PRICE_PROJECT",
        schema_name="ML_LAYER"
    )

    return registry.get_model("PREPROCESSING_PIPELINE").show_versions().shape[0]

In [17]:
action_nr = 1
for action, check in actions_and_checks:
    print(f"\n({action_nr}/7) {action} ********************")
    action_nr += 1
    match action:

        case "sql_ingest":
            try:          
                print(session.sql(sql_ingest.strip()).collect())
            except:
                print(f"{action} - failed")
                break
            
            if not TaskSuccessful(check):
                break

        case "ML_LAYER.PREPROCESSING_PROC()":
            try:
                print(session.sql(f"CALL HOUSING_PRICE_PROJECT.{action};").collect())
            except:
                print(f"{action} - failed")
                break            
            
            if not TaskSuccessful(check):
                break

        case _:
            try:
                print(session.sql(f"CALL HOUSING_PRICE_PROJECT.{action};").collect())
            except:
                print(f"{action} - failed")
                break
                
            if not TaskSuccessful(check):
                break


sql_ingest ********************
[Row(file='s3://snowflake-housingprice-ml-pipeline-project/HousingPricePredictionDataset_train.csv', status='LOADED', rows_parsed=35000, rows_loaded=35000, error_limit=1, errors_seen=0, first_error=None, first_error_line=None, first_error_character=None, first_error_column_name=None)]

STAGING_LAYER.QUALITY_CHECK_REPORT_PROC() ********************
[Row(QUALITY_CHECK_REPORT_PROC=None)]

CLEANING_LAYER.CLEAN_DATA_PROC(10) ********************
[Row(CLEAN_DATA_PROC='[Row(number of rows inserted=35000)]')]

ML_LAYER.TRAINTEST_SPLIT_PROC(5000) ********************
[Row(TRAINTEST_SPLIT_PROC=None)]

ML_LAYER.PREPROCESSING_PROC() ********************
[Row(PREPROCESSING_PROC=None)]

ML_LAYER.HYPERPARAM_TUNING_PROC() ********************
[Row(HYPERPARAM_TUNING_PROC='DONE')]

ML_LAYER.XGBOOST_TRAINING_PROC() ********************
[Row(XGBOOST_TRAINING_PROC='DONE')]


1


# Tasks Activation

In [39]:
# Pipe Activation
session.sql("""ALTER PIPE HOUSING_PRICE_PROJECT.STAGING_LAYER.S3_INGESTION_PIPE
SET PIPE_EXECUTION_PAUSED = FALSE;""").collect()

[Row(status='Statement executed successfully.')]

In [36]:
# Activate Tasks-Tree (i.e. all tasks except for "HYPERPARAM_TUNING_TASK")
session.sql("""SELECT SYSTEM$TASK_DEPENDENTS_ENABLE(
    'HOUSING_PRICE_PROJECT.PUBLIC.QUALITYCHECK_REPORT_TASK'
);""").collect()

[Row(SYSTEM$TASK_DEPENDENTS_ENABLE(
     'HOUSING_PRICE_PROJECT.PUBLIC.QUALITYCHECK_REPORT_TASK'
 )='OK')]

In [37]:
# Activate task "HYPERPARAM_TUNING_TASK"
session.sql("""ALTER TASK HOUSING_PRICE_PROJECT.PUBLIC.HYPERPARAM_TUNING_TASK RESUME;""").collect()

[Row(status='Statement executed successfully.')]